# STAR-RIS RSMA — v19: Scalability theo N (Chứng minh lợi thế phân rã MADDPG)
## Notebook bổ trợ — củng cố §4.7 & Bảng 4.6 ("Khả năng mở rộng: Cao") của luận văn

---

### 📋 Cập nhật so với v18 (vì sao có v19?)
> **v18 đã đạt:** convergence rõ (freeze λ), MADDPG SumRate **3.44±0.08** (CI hẹp nhất), latency vs BCD **13×**.
> **NHƯNG tại N=32, MADDPG ≈ TD3** (3.44 vs 3.38, chưa significant). Luận điểm "MADDPG ổn định hơn" còn yếu.
>
> **v19 lấp đúng khoảng trống này:** lý thuyết CTDE (Lowe 2017) nói lợi thế phân rã **tăng theo kích thước
> không gian hành động**. Ở bài toán này, hành động đơn-agent ≈ `3N+9` chiều (N=16→57, N=32→105, N=64→201).
> Khi N lớn, đơn-agent (TD3/DDPG) phải khám phá không gian khổng lồ → khó; MADDPG tách thành `[9, 2N, N]`
> nên tăng chậm hơn → **MADDPG phải tách top khi N tăng**. v19 sweep N để KIỂM CHỨNG điều này bằng số.

| Hạng mục | v18 (demo chính) | **v19 (scalability)** |
|---|---|---|
| Mục tiêu | Convergence + số MADDPG tốt nhất | **Bằng chứng MADDPG mở rộng tốt theo N** |
| Thuật toán | MADDPG/DDPG/TD3/PPO | **MADDPG vs TD3** (đối đầu trực tiếp) |
| N | cố định 32 | **quét {16, 32, 64}** |
| Config λ/residual | freeze 0.55, λ_max 13, residual 0.25 | **GIỮ NGUYÊN như v18** (công bằng) |
| Seeds × episodes | 5 × 2000 | 3 × 1500 (đủ thấy xu hướng, vừa compute) |

### 🎯 Kết quả MONG ĐỢI (để củng cố luận văn)
1. **`scal_sumrate_vs_N.png`** — đường MADDPG **vượt dần** TD3 khi N tăng (ở N=16 có thể ngang/thua, N=64 dẫn rõ).
2. **`scal_decomposition_gain.png`** — đường `gain(N) = SR(MADDPG) − SR(best single-agent)` **ĐI LÊN theo N** (≥0 và lớn dần). ⭐ Đây là hình quan trọng nhất.
3. **`tables/scalability_vs_N.csv`** + dòng in tự động: *"gain tăng từ … (N=16) → … (N=64) ⇒ củng cố lý thuyết phân rã CTDE"*.

> ✅ **Nếu gain↑ theo N:** bạn viết được vào luận văn — *"lợi thế MADDPG so với đơn-agent tăng theo số phần tử RIS,
> khẳng định giá trị kiến trúc phân rã CTDE khi không gian hành động lớn"*. Đặt cạnh latency 13× vs BCD → câu chuyện
> hoàn chỉnh: **gần tối ưu · độ trễ thấp · mở rộng tốt**.
>
> ⚠️ **Nếu gain phẳng/âm:** báo cáo trung thực (đúng tinh thần luận văn) — ở dải N này MADDPG ngang đơn-agent;
> điểm bán là thực thi phi tập trung + latency. Cân nhắc thêm N=96/128 để lộ xu hướng (chỉ sửa `N_LIST`).

### ⚙️ BẮT BUỘC trước khi chạy
- **Chạy ở SESSION KAGGLE RIÊNG** (đừng gộp v18 — sẽ quá giới hạn 12h).
- Dùng **cùng project dataset như v18** (phải có `experiments/train.py` đã thêm freeze λ).
- Sửa `PROJECT_ROOT` ở cell import cho khớp tên dataset bạn upload.

### ⏱️ Thời gian: cell config (mục 3) sẽ in ước lượng. Nếu > 11h → giảm `N_LIST` hoặc `SCAL_SEEDS`.


### 1. Kiểm tra phần cứng & cài thư viện

In [ ]:
import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
!pip install gymnasium pyyaml tqdm pandas matplotlib tensorboard

### 2. Đường dẫn dự án & import

In [ ]:
import os, sys, copy, time
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from IPython.display import Image, display

# ⚠️ Sửa cho khớp tên dataset bạn upload lên Kaggle:
PROJECT_ROOT = "/kaggle/input/datasets/duythanhb1909984/star-ris-rsma-maddpg-v14"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from experiments.train import train_maddpg, train_single_agent
from experiments.evaluate import _eval_multi_seed
from utils.plotting import plot_metric_vs_x
from utils import confidence_interval
print("Import OK")

### 3. Cấu hình v18 + tham số scalability

**📌 Output mong đợi (in từ cell dưới):**
- Xác nhận config v18: `freeze=0.55, λ_max=13, target=0.50, residual_scale=0.25`.
- `N_LIST = [16, 32, 64]`, `Algos = ['maddpg','td3']`, `Seeds = [1000,2000,3000]`, `Episodes = 1500`.
- `Số lần train = 18` và **ước lượng thời gian (giờ)** — kiểm tra < 11h trước khi Run All.

> Chỉnh `N_LIST` / `SCAL_SEEDS` / `SCAL_EPISODES` / `SCAL_ALGOS` ở cell code tùy ngân sách. Thêm `"ddpg"` hoặc N=96 nếu còn giờ.


In [ ]:
config_path = os.path.join(PROJECT_ROOT, "config", "config.yaml")
with open(config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# ----- Đồng bộ tham số v18 (freeze λ + cân bằng QoS) để công bằng với notebook chính -----
cfg["env"]["qos_lambda_freeze_fraction"] = 0.55
cfg["env"]["qos_lambda_max"] = 13.0
cfg["env"]["qos_target_satisfaction"] = 0.50
cfg["env"]["qos_lambda_increase"] = 1.02
cfg["env"]["phase_residual_scale"] = 0.25

# ----- Tham số scalability (điều chỉnh theo ngân sách Kaggle) -----
N_LIST        = [16, 32, 64]              # số phần tử STAR-RIS để quét
SCAL_ALGOS    = ["maddpg", "td3"]         # head-to-head; thêm "ddpg" nếu còn giờ
SCAL_SEEDS    = [1000, 2000, 3000]        # 3 seeds đủ thấy xu hướng
SCAL_EPISODES = 1500                      # đủ hội tụ (curve phẳng ~ep 1000)

LABEL = {"maddpg": "MADDPG", "td3": "TD3", "ddpg": "DDPG"}

out_root = "/kaggle/working/"
fig_dir  = os.path.join(out_root, "figures")
tab_dir  = os.path.join(out_root, "tables")
log_dir  = os.path.join(out_root, "logs_scal")
ckpt_dir = os.path.join(out_root, "checkpoints_scal")
for d in (fig_dir, tab_dir):
    os.makedirs(d, exist_ok=True)

# Ước lượng thời gian (thô, theo nhịp ~ N=32, 2000ep: MADDPG~38ph, TD3~14ph/seed)
_base = {"maddpg": 38.0, "td3": 14.0, "ddpg": 14.0}
est_min = 0.0
for N in N_LIST:
    scale_N = (N / 32.0)            # chi phí ~ tuyến tính theo N (action-dim)
    for a in SCAL_ALGOS:
        est_min += _base[a] * (SCAL_EPISODES / 2000.0) * max(scale_N, 0.5) * len(SCAL_SEEDS)
print("="*60)
print("  SCALABILITY SWEEP — cấu hình")
print("="*60)
print(f"  N_LIST          = {N_LIST}")
print(f"  Algos           = {SCAL_ALGOS}")
print(f"  Seeds           = {SCAL_SEEDS}")
print(f"  Episodes/seed   = {SCAL_EPISODES}")
print(f"  Số lần train    = {len(N_LIST)*len(SCAL_ALGOS)*len(SCAL_SEEDS)}")
print(f"  λ: freeze={cfg['env']['qos_lambda_freeze_fraction']}, max={cfg['env']['qos_lambda_max']}, "
      f"residual_scale={cfg['env']['phase_residual_scale']}")
print(f"  ⏱️  Ước lượng    ~ {est_min/60.0:.1f} giờ (T4). Nếu > 11h: bỏ bớt N hoặc seed!")
print("="*60)

### 4. Huấn luyện lại tại mỗi N (MADDPG vs single-agent)

**📌 Output mong đợi:** với mỗi `N`, tqdm bar cho từng (algo × seed). Mất nhiều giờ — đây là cell nặng nhất.

> Mỗi N: deep-copy config, đặt `num_ris_elements=N`, huấn luyện từ đầu (agent build theo `env.spec()`),
> rồi đánh giá đa-seed với λ huấn luyện-cuối. Kết quả lưu vào `scal_results`.


In [ ]:
scal_results = {}      # scal_results[N][label] = {"sr_mean","sr_ci","qos_mean","qos_ci","srs"}
rows = []

for N in N_LIST:
    cfg_N = copy.deepcopy(cfg)
    cfg_N["env"]["num_ris_elements"] = int(N)
    cfg_N["training"]["total_episodes"] = int(SCAL_EPISODES)
    print(f"\n{'#'*64}\n#  N = {N}  (single-agent action-dim ~ {3*N + 9})\n{'#'*64}")
    per_algo = {}
    for algo in SCAL_ALGOS:
        label = LABEL[algo]
        srs, qoss = [], []
        for s in SCAL_SEEDS:
            run = f"scal_{algo}_N{N}_s{s}"
            print(f"\n--- Train {label} | N={N} | seed={s} ---")
            if algo == "maddpg":
                info = train_maddpg(cfg_N, log_dir=log_dir, ckpt_dir=ckpt_dir,
                                    seed_override=s, run_name=run)
            else:
                info = train_single_agent(cfg_N, kind=algo, log_dir=log_dir, ckpt_dir=ckpt_dir,
                                          seed_override=s, run_name=run)
            lam = info.get("trained_qos_lambda")
            m = _eval_multi_seed(info["agent"], algo, info["obs_norm"], cfg_N,
                                 cfg_N["evaluation"]["seeds"], qos_lambda=lam)
            srs.append(m["sum_rate_mean"]); qoss.append(m["qos_mean"])
        sr_m, sr_ci, _ = confidence_interval(np.array(srs))
        q_m,  q_ci, _  = confidence_interval(np.array(qoss))
        per_algo[label] = {"sr_mean": sr_m, "sr_ci": sr_ci,
                           "qos_mean": q_m, "qos_ci": q_ci, "srs": srs}
        rows.append({"N": N, "Algorithm": label,
                     "SumRate": sr_m, "SumRate_CI": sr_ci,
                     "QoS": q_m, "QoS_CI": q_ci, "N_seeds": len(SCAL_SEEDS)})
        print(f"  => {label} N={N}: SR={sr_m:.3f}±{sr_ci:.3f}, QoS={q_m:.3f}±{q_ci:.3f}")
    scal_results[N] = per_algo

df_scal = pd.DataFrame(rows)
df_scal.to_csv(os.path.join(tab_dir, "scalability_vs_N.csv"), index=False)
print("\n--- Bảng scalability (scalability_vs_N.csv) ---")
display(df_scal)

### 5. Đồ thị & phân tích lợi thế phân rã

**📌 Output mong đợi:** 3 biểu đồ — (1) SR vs N, (2) QoS vs N, (3) **Decomposition gain vs N**.

**🎯 Bằng chứng cần có:** đường *gain* (MADDPG − best single-agent) **đi lên theo N**.
Nếu đúng ⇒ viết được: *"lợi thế của MADDPG so với đơn-agent tăng theo số phần tử RIS, khẳng định giá trị
của kiến trúc phân rã CTDE khi không gian hành động lớn."*


In [ ]:
labels = [LABEL[a] for a in SCAL_ALGOS]

# (1) Sum-rate vs N
sr_curves  = {lab: {"mean": [scal_results[N][lab]["sr_mean"] for N in N_LIST],
                    "ci":   [scal_results[N][lab]["sr_ci"]   for N in N_LIST]} for lab in labels}
plot_metric_vs_x(N_LIST, sr_curves, xlabel="STAR-RIS elements $N$",
                 ylabel="Avg. sum-rate (b/s/Hz)", out_dir=fig_dir, name="scal_sumrate_vs_N")

# (2) QoS vs N
qos_curves = {lab: {"mean": [scal_results[N][lab]["qos_mean"] for N in N_LIST],
                    "ci":   [scal_results[N][lab]["qos_ci"]   for N in N_LIST]} for lab in labels}
plot_metric_vs_x(N_LIST, qos_curves, xlabel="STAR-RIS elements $N$",
                 ylabel="QoS satisfaction probability", out_dir=fig_dir, name="scal_qos_vs_N")

display(Image(filename=os.path.join(fig_dir, "scal_sumrate_vs_N.png")))
display(Image(filename=os.path.join(fig_dir, "scal_qos_vs_N.png")))

# (3) Decomposition gain = SR(MADDPG) - SR(best single-agent) theo N
if "MADDPG" in labels and len(labels) > 1:
    single = [l for l in labels if l != "MADDPG"]
    gains, gain_labels = [], []
    for N in N_LIST:
        sr_madd = scal_results[N]["MADDPG"]["sr_mean"]
        best_single = max(scal_results[N][l]["sr_mean"] for l in single)
        gains.append(sr_madd - best_single); gain_labels.append(str(N))
    plt.figure(figsize=(5.8, 3.7))
    plt.axhline(0.0, color="gray", lw=0.8, ls="--")
    plt.plot(N_LIST, gains, marker="o", color="#1f77b4", lw=2)
    plt.fill_between(N_LIST, 0, gains, alpha=0.15, color="#1f77b4")
    plt.xlabel("STAR-RIS elements $N$")
    plt.ylabel("Decomposition gain\nSR(MADDPG) − SR(best single-agent)")
    plt.title("Lợi thế MADDPG tăng theo N (kỳ vọng đi lên)")
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "scal_decomposition_gain.png"), dpi=150)
    plt.show()

    print("\n--- Decomposition gain theo N ---")
    for N, g in zip(N_LIST, gains):
        flag = "✅ MADDPG dẫn" if g > 0 else "⚠️ chưa dẫn"
        print(f"  N={N:3d}: gain = {g:+.3f} b/s/Hz   {flag}")
    if len(gains) >= 2 and gains[-1] > gains[0]:
        print(f"\n  ✅ XU HƯỚNG ĐÚNG: gain tăng từ {gains[0]:+.3f} (N={N_LIST[0]}) -> {gains[-1]:+.3f} (N={N_LIST[-1]})")
        print("     => Củng cố lý thuyết: lợi thế phân rã CTDE tăng theo không gian hành động.")
    else:
        print("\n  ⚠️ Gain CHƯA tăng theo N. Cân nhắc: thêm N lớn hơn (96/128), hoặc nhấn thế mạnh khác"
              " (latency, decentralized execution).")

### 6. Diễn giải cho bài báo

- **Nếu gain tăng theo N** → đây là *đóng góp định lượng* mạnh nhất cho việc chọn MADDPG:
  *"Khi N tăng (không gian hành động RIS phình to), MADDPG vượt dần đơn-agent nhờ phân rã CTDE."*
  Đặt cạnh kết quả latency (MADDPG ~13× nhanh hơn BCD) → câu chuyện hoàn chỉnh: **gần tối ưu, độ trễ thấp,
  và mở rộng tốt theo quy mô RIS.**
- **Nếu gain phẳng/âm** → trung thực: ở dải N này MADDPG ngang đơn-agent; điểm bán là **thực thi phi tập trung
  (decentralized execution)** + latency, không phải sum-rate. Cân nhắc N lớn hơn để lộ xu hướng.
